# 🍼 Infant Cry Classifier — Google Colab v3
### Production Pipeline · 3-Stage · Temperature Calibrated · 87.50% Ensemble

| Class | Meaning |
|-------|---------|
| `belly_pain` | Stomach / colic pain |
| `burping`    | Needs to burp |
| `discomfort` | Wet diaper / temperature |
| `hungry`     | Hunger cry |
| `tired`      | Overtired / sleepy |

## 🔒 Production 3-Stage Pipeline
```
Audio Input
     │
     ▼
┌─────────────────────────────────────────────┐
│  STAGE 1: Custom BinaryCryCNN Gate          │
│  "Is this even a baby cry?" (98.56% acc)   │
│  Trained on 4000 cry + 2000 ESC-50 samples  │
└─────────────────────────────────────────────┘
     │ is_cry = True              │ REJECTED
     ▼                            ▼
┌──────────────────────┐   🚫 NOT A BABY CRY
│  STAGE 2: OOD Check  │   (music/noise/speech)
│  Entropy ≤ 1.35 AND  │
│  Confidence ≥ 42%    │
└──────────────────────┘
     │ accepted
     ▼
┌──────────────────────────────────────────────────┐
│  STAGE 3: 5-Class Calibrated Ensemble             │
│  BaselineCNN · CNN+BiLSTM · CryNet · SE-ResNet    │
│  Temperature-scaled logits + learned weights      │
│  SE-ResNet 36% · CryNet 34% · BiLSTM 16% · CNN 14%│
└──────────────────────────────────────────────────┘
     │
     ▼
belly_pain / burping / discomfort / hungry / tired
```

### ⚡ Quick Start
1. ▶ **Cell 1** — Install libraries (~60 sec first run)
2. ▶ **Cell 2** — Mount Google Drive
3. ▶ **Cell 3** — Set `MODEL_ROOT` to your Drive folder
4. ▶ **Cell 4** — Load calibration artifacts (temperature.json, ensemble_weights.json)
5. ▶ **Cell 5** — Paste calibration values (offline mode — no Drive needed)
6. ▶ **Cell 6** — Model architectures (all 4 + BinaryCryCNN gate)
7. ▶ **Cell 7** — Load models from Drive
8. ▶ **Cell 8** — Preprocessing functions
9. ▶ **Cell 9** — Upload audio → get prediction
10. ▶ **Cell 10** — Batch inference on folder

> 📁 Copy from `D:\nxm\ML_pipeline\models\` → Google Drive `cry-models/`  
> Required: `cry_gate/best_model.pth`, `baseline_cnn/dataset1_augmented/best_model.pth`,  
> `cnn_bilstm/dataset1_augmented/best_model.pth`, `cnn_transformer/dataset1_augmented/best_model.pth`,  
> `se_resnet/dataset1_augmented/best_model.pth`  
> Also: `training/features/dataset1/norm_stats.json`  
> Calibration JSON files are **embedded inline** in Cell 5 (no Drive needed for those)


In [ ]:
# ── Cell 1: Install Dependencies ────────────────────────────────────────────
import subprocess, sys, os

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

subprocess.run([sys.executable, "-m", "pip", "install",
                "librosa==0.11.0", "soundfile", "-q"], check=True)

import torch, numpy as np, librosa

ret = subprocess.run(["ffmpeg", "-version"], capture_output=True)
ffmpeg_ok = ret.returncode == 0

print("=" * 52)
print("  ✅ Dependencies ready")
print(f"  PyTorch  : {torch.__version__}")
print(f"  librosa  : {librosa.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")
print(f"  ffmpeg   : {'✅ mp3 supported' if ffmpeg_ok else '⚠️ wav only'}")
if torch.cuda.is_available():
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
print("=" * 52)


In [ ]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive/")


In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
import os, torch
from pathlib import Path

# ⚙️ SET THIS to your Google Drive folder containing model .pth files
# Example: /content/drive/MyDrive/cry-models
MODEL_ROOT = "/content/drive/MyDrive/cry-models"

# Path to norm_stats.json (saved from training features)
NORM_STATS_PATH = f"{MODEL_ROOT}/norm_stats.json"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Config set")
print(f"  MODEL_ROOT : {MODEL_ROOT}")
print(f"  Device     : {DEVICE}")
print(f"  norm_stats : {NORM_STATS_PATH}")

# Verify folders exist
model_paths = {
    'cry_gate':       f"{MODEL_ROOT}/cry_gate/best_model.pth",
    'baseline_cnn':   f"{MODEL_ROOT}/baseline_cnn/dataset1_augmented/best_model.pth",
    'cnn_bilstm':     f"{MODEL_ROOT}/cnn_bilstm/dataset1_augmented/best_model.pth",
    'cnn_transformer':f"{MODEL_ROOT}/cnn_transformer/dataset1_augmented/best_model.pth",
    'se_resnet':      f"{MODEL_ROOT}/se_resnet/dataset1_augmented/best_model.pth",
}
print()
for name, path in model_paths.items():
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "MISSING"
    print(f"  {'✅' if exists else '❌'} {name:20s}: {size}")


In [ ]:
# ── Cell 4a: Load Calibration from Drive (if you have them there) ────────────
# If you don't have these files on Drive, skip to Cell 4b (inline values)
import json, os

CALIB_DIR = f"{MODEL_ROOT}/calibration"
temp_path = f"{CALIB_DIR}/temperature.json"
wt_path   = f"{CALIB_DIR}/ensemble_weights.json"

CALIB_TEMPERATURES = None
CALIB_WEIGHTS      = None

if os.path.exists(temp_path) and os.path.exists(wt_path):
    with open(temp_path) as f: CALIB_TEMPERATURES = json.load(f)['temperatures']
    with open(wt_path)   as f: CALIB_WEIGHTS      = json.load(f)['weights']
    print("✅ Calibration loaded from Drive")
    print(f"  Temperatures : {CALIB_TEMPERATURES}")
    print(f"  Weights      : {CALIB_WEIGHTS}")
else:
    print("⚠️  Calibration JSONs not found on Drive → Run Cell 4b to use inline values")


In [ ]:
# ── Cell 4b: Inline Calibration Values (always safe to run) ─────────────────
# These are the Phase 4 calibration values computed from your training run.
# Run this cell whether or not Drive has the JSON files.

CALIB_TEMPERATURES = {
    "BaselineCNN*" : 0.873054,   # overconfident → T<1 sharpens
    "CNN+BiLSTM*"  : 0.951187,   # slight overconfidence
    "CryNet*"      : 1.001823,   # near-perfect calibration
    "SE-ResNet*"   : 1.037412,   # slightly underconfident → T>1 softens
}

CALIB_WEIGHTS = {
    "BaselineCNN*" : 0.137214,   # 13.7%
    "CNN+BiLSTM*"  : 0.162346,   # 16.2%
    "CryNet*"      : 0.341028,   # 34.1%
    "SE-ResNet*"   : 0.359412,   # 36.0%
}

CLASSES = ['belly_pain', 'burping', 'discomfort', 'hungry', 'tired']

print("✅ Calibration values set (Phase 4 — Temperature Scaling + Learned Weights)")
print()
print("  Temperatures (T<1 = was overconfident, T>1 = was underconfident):")
for name, T in CALIB_TEMPERATURES.items():
    marker = "← overconf" if T < 0.99 else ("← underconf" if T > 1.01 else "← calibrated")
    print(f"    {name:15s}: T={T:.4f}  {marker}")
print()
print("  Ensemble Weights (learned via Nelder-Mead optimisation):")
for name, w in CALIB_WEIGHTS.items():
    bar = '█' * int(w * 100 // 5)
    print(f"    {name:15s}: {w*100:5.1f}%  {bar}")


In [ ]:
# ── Cell 5: Model Architecture Definitions ───────────────────────────────────
# These definitions MUST match the training scripts exactly.
import torch, torch.nn as nn, torch.nn.functional as F, random

class SpecAugment(nn.Module):
    def __init__(self, freq_mask=15, time_mask=40, nf=2, nt=2):
        super().__init__()
        self.fm = freq_mask; self.tm = time_mask; self.nf = nf; self.nt = nt
    def forward(self, x):
        if not self.training: return x
        B, C, F, T = x.shape
        x = x.clone()
        for _ in range(self.nf):
            f = random.randint(0, self.fm); f0 = random.randint(0, max(1, F-f))
            x[:,:,f0:f0+f,:] = 0.0
        for _ in range(self.nt):
            t = random.randint(0, self.tm); t0 = random.randint(0, max(1, T-t))
            x[:,:,:,t0:t0+t] = 0.0
        return x

# ── Binary Cry Gate CNN ───────────────────────────────────────────────────────
class _GateResBlock(nn.Module):
    def __init__(self, inc, outc, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(inc, outc, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(outc), nn.ReLU(inplace=True),
            nn.Conv2d(outc, outc, 3, padding=1, bias=False), nn.BatchNorm2d(outc))
        self.skip = nn.Sequential(
            nn.Conv2d(inc, outc, 1, stride=stride, bias=False), nn.BatchNorm2d(outc)
        ) if (inc != outc or stride != 1) else nn.Identity()
    def forward(self, x):
        return F.relu(self.conv(x) + self.skip(x), True)

class BinaryCryCNN(nn.Module):
    """Lightweight binary cry detector: cry vs not-cry. ~37k params."""
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16),
            nn.ReLU(True), nn.MaxPool2d(2), nn.Dropout2d(0.1))
        self.r1   = _GateResBlock(16, 32, stride=2)
        self.r2   = _GateResBlock(32, 64, stride=2)
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x):
        x = self.stem(x); x = self.r1(x); x = self.r2(x)
        return self.head(self.gap(x))

# ── Shared CNN Encoder (BiLSTM & CryNet) ─────────────────────────────────────
class _EncResBlock(nn.Module):
    def __init__(self, ch, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch,ch,3,padding=1,bias=False), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(ch,ch,3,padding=1,bias=False), nn.BatchNorm2d(ch))
        self.relu = nn.ReLU(True)
    def forward(self, x): return self.relu(self.net(x) + x)

class CNNEncoder(nn.Module):
    def __init__(self, out_ch=256):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1,32,3,padding=1,bias=False), nn.BatchNorm2d(32), nn.ReLU(True))
        self.b1 = nn.Sequential(_EncResBlock(32), nn.MaxPool2d(2,2))
        self.b2 = nn.Sequential(nn.Conv2d(32,64,3,padding=1,bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
                                _EncResBlock(64), nn.MaxPool2d(2,2))
        self.b3 = nn.Sequential(nn.Conv2d(64,128,3,padding=1,bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
                                _EncResBlock(128), nn.MaxPool2d(2,2))
        self.b4 = nn.Sequential(nn.Conv2d(128,out_ch,3,padding=1,bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
                                _EncResBlock(out_ch), nn.MaxPool2d(2,2))
        self.fpool = nn.AdaptiveAvgPool2d((1, None))
    def forward(self, x):
        x = self.stem(x); x = self.b1(x); x = self.b2(x); x = self.b3(x); x = self.b4(x)
        return self.fpool(x).squeeze(2).transpose(1,2)

class TemporalAttn(nn.Module):
    def __init__(self, h):
        super().__init__(); self.a = nn.Linear(h*2, 1)
    def forward(self, x):
        w = F.softmax(self.a(x).squeeze(-1), 1)
        return (w.unsqueeze(-1) * x).sum(1), w

# ── BaselineCNN ───────────────────────────────────────────────────────────────
class _CnnResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,out_ch,3,stride=stride,padding=1,bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop = nn.Dropout2d(0.1)
        self.skip = nn.Sequential(nn.Conv2d(in_ch,out_ch,1,stride=stride,bias=False),
                                  nn.BatchNorm2d(out_ch)) if (stride!=1 or in_ch!=out_ch) else nn.Sequential()
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x))); out = self.drop(out)
        out = self.bn2(self.conv2(out)); return F.relu(out + self.skip(x))

class BaselineCNN(nn.Module):
    def __init__(self, n_classes=5, dropout=0.5):
        super().__init__()
        self.spec_aug = SpecAugment(freq_mask=15, time_mask=40)
        self.stem = nn.Sequential(
            nn.Conv2d(1,32,kernel_size=7,stride=2,padding=3,bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3,stride=2,padding=1), nn.Dropout2d(0.1))
        self.stage1=_CnnResBlock(32,64,stride=2); self.stage2=_CnnResBlock(64,128,stride=2)
        self.stage3=_CnnResBlock(128,256,stride=2); self.stage4=_CnnResBlock(256,256,stride=1)
        self.gap=nn.AdaptiveAvgPool2d(1); self.drop=nn.Dropout(dropout)
        self.fc1=nn.Linear(256,128); self.bn_fc=nn.BatchNorm1d(128); self.fc2=nn.Linear(128,n_classes)
    def forward(self, x):
        x=self.spec_aug(x); x=self.stem(x)
        x=self.stage1(x); x=self.stage2(x); x=self.stage3(x); x=self.stage4(x)
        x=self.gap(x).flatten(1); x=self.drop(x)
        return self.fc2(self.drop(F.relu(self.bn_fc(self.fc1(x)))))

# ── CNN+BiLSTM ────────────────────────────────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, n=5, h=256, layers=2):
        super().__init__()
        self.aug=SpecAugment(); self.enc=CNNEncoder(256)
        self.lstm=nn.LSTM(256,h,layers,batch_first=True,bidirectional=True,dropout=0.3 if layers>1 else 0.0)
        self.attn=TemporalAttn(h)
        self.head=nn.Sequential(nn.LayerNorm(h*2),nn.Linear(h*2,256),nn.GELU(),nn.Dropout(0.4),
                                nn.Linear(256,64),nn.GELU(),nn.Dropout(0.2),nn.Linear(64,n))
    def forward(self, x):
        x=self.aug(x); x=self.enc(x); out,_=self.lstm(x); ctx,_=self.attn(out)
        return self.head(ctx)

# ── CryNet (Transformer) ──────────────────────────────────────────────────────
class LearnedPE(nn.Module):
    def __init__(self, maxlen, d, drop=0.1):
        super().__init__(); self.pe=nn.Embedding(maxlen,d); self.drop=nn.Dropout(drop)
    def forward(self, x):
        B,T,D=x.shape; pos=torch.arange(T,device=x.device).unsqueeze(0).expand(B,-1)
        return self.drop(x + self.pe(pos))

class CryNet(nn.Module):
    def __init__(self, n=5, d=256, heads=8, layers=4, ff=512, drop=0.2):
        super().__init__()
        self.aug=SpecAugment(freq_mask=20,time_mask=50); self.enc=CNNEncoder(d)
        self.pe=LearnedPE(128,d,drop); self.cls=nn.Parameter(torch.zeros(1,1,d))
        nn.init.trunc_normal_(self.cls,std=0.02)
        tfl=nn.TransformerEncoderLayer(d,heads,ff,drop,activation='gelu',batch_first=True,norm_first=True)
        self.tf=nn.TransformerEncoder(tfl,layers,norm=nn.LayerNorm(d))
        self.head=nn.Sequential(nn.LayerNorm(d),nn.Linear(d,256),nn.GELU(),nn.Dropout(0.4),
                                nn.Linear(256,64),nn.GELU(),nn.Dropout(0.2),nn.Linear(64,n))
    def forward(self, x):
        x=self.aug(x); x=self.enc(x); x=self.pe(x)
        cls=self.cls.expand(x.size(0),-1,-1); x=self.tf(torch.cat([cls,x],1))
        return self.head(x[:,0])

# ── SE-ResNet ─────────────────────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.sq=nn.AdaptiveAvgPool2d(1)
        self.ex=nn.Sequential(nn.Flatten(),nn.Linear(c,max(c//r,8)),nn.ReLU(inplace=True),
                              nn.Linear(max(c//r,8),c),nn.Sigmoid())
    def forward(self, x): return x * self.ex(self.sq(x)).view(x.size(0),x.size(1),1,1)

class SEResBlock(nn.Module):
    def __init__(self, inc, outc, stride=1):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(inc,outc,3,stride=stride,padding=1,bias=False),
                                nn.BatchNorm2d(outc),nn.ReLU(inplace=True),
                                nn.Conv2d(outc,outc,3,padding=1,bias=False),nn.BatchNorm2d(outc))
        self.se=SEBlock(outc)
        self.skip=nn.Sequential(nn.Conv2d(inc,outc,1,stride=stride,bias=False),
                                nn.BatchNorm2d(outc)) if (inc!=outc or stride!=1) else nn.Identity()
    def forward(self, x): return F.relu(self.se(self.conv(x)) + self.skip(x), True)

class SEResNet(nn.Module):
    def __init__(self, n=5):
        super().__init__()
        self.aug=SpecAugment(); self.stem=nn.Sequential(nn.Conv2d(1,32,3,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU(inplace=True))
        self.layer1=nn.Sequential(SEResBlock(32,64,stride=2),SEResBlock(64,64))
        self.layer2=nn.Sequential(SEResBlock(64,128,stride=2),SEResBlock(128,128))
        self.layer3=nn.Sequential(SEResBlock(128,256,stride=2),SEResBlock(256,256))
        self.layer4=nn.Sequential(SEResBlock(256,512,stride=2),SEResBlock(512,512))
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Sequential(nn.Flatten(),nn.Dropout(0.4),nn.Linear(512,256),nn.GELU(),nn.Dropout(0.3),nn.Linear(256,n))
    def forward(self, x):
        x=self.stem(self.aug(x)); x=self.layer1(x); x=self.layer2(x); x=self.layer3(x); x=self.layer4(x)
        return self.head(self.pool(x))

print("✅ All model architectures defined:")
print("  BinaryCryCNN  — binary cry gate (~37k params)")
print("  BaselineCNN   — 4-stage residual CNN (2.4M params)")
print("  CNNBiLSTM     — CNN encoder + BiLSTM + attention")
print("  CryNet        — CNN encoder + 4-layer Transformer")
print("  SEResNet      — 4-layer SE-ResNet (best aug: +3.48%)")


In [ ]:
# ── Cell 6: Load Models from Google Drive ────────────────────────────────────
import os, torch

def load_checkpoint(model, ckpt_path, name):
    if not os.path.exists(ckpt_path):
        print(f"  ❌ {name}: {ckpt_path} NOT FOUND")
        return None
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    model.eval().to(DEVICE)
    val_acc = ckpt.get('val_acc', 0.0)
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print(f"  ✅ {name:20s}: val_acc={val_acc:.4f}  ({size_mb:.1f} MB)")
    return model

# Load binary gate
_gate_model = load_checkpoint(BinaryCryCNN(), model_paths['cry_gate'], 'BinaryCryCNN (gate)')

# Load 5-class ensemble models
_model_registry = [
    (BaselineCNN(n_classes=5),       model_paths['baseline_cnn'],    'BaselineCNN*'),
    (CNNBiLSTM(n=5),                 model_paths['cnn_bilstm'],      'CNN+BiLSTM*'),
    (CryNet(n=5),                    model_paths['cnn_transformer'],  'CryNet*'),
    (SEResNet(n=5),                  model_paths['se_resnet'],        'SE-ResNet*'),
]
_loaded_models = []
print()
print("Loading 5-class ensemble models:")
for model_obj, ckpt_path, name in _model_registry:
    loaded = load_checkpoint(model_obj, ckpt_path, name)
    if loaded is not None:
        _loaded_models.append((loaded, name))

print()
print(f"  {len(_loaded_models)}/4 ensemble models loaded")
if len(_loaded_models) == 0:
    print("  ❌ No models loaded — check MODEL_ROOT path in Cell 3")


In [ ]:
# ── Cell 7: Load Normalisation Statistics ────────────────────────────────────
import json, numpy as np

NORM_STATS = None
if os.path.exists(NORM_STATS_PATH):
    with open(NORM_STATS_PATH) as f:
        NORM_STATS = json.load(f)
    print(f"✅ norm_stats.json loaded from {NORM_STATS_PATH}")
    print(f"  mel_mean shape : {len(NORM_STATS['mel_mean'])} bins")
    print(f"  mel_std  shape : {len(NORM_STATS['mel_std'])} bins")
else:
    print(f"❌ norm_stats.json not found at: {NORM_STATS_PATH}")
    print("   Copy it from D:\\nxm\\ML_pipeline\\training\\features\\dataset1\\norm_stats.json")


In [ ]:
# ── Cell 8: Preprocessing Functions ─────────────────────────────────────────
import librosa, numpy as np, torch

# ── Audio constants — MUST match training ────────────────────────────────────
SR         = 22050
DURATION   = 10.0
N_SAMPLES  = int(SR * DURATION)   # 220500
N_MEL      = 128
N_FFT      = 2048
HOP_LEN    = 512
FMIN       = 50
FMAX       = 8000
TARGET_T   = 431

GATE_THRESHOLD   = 0.34   # F1-optimised binary gate threshold
OOD_CONF_THR     = 0.42   # minimum confidence to accept
OOD_ENTROPY_THR  = 1.35   # maximum entropy to accept

def preprocess_audio(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    if len(audio) == 0: raise ValueError(f"Empty audio: {path}")
    peak = np.max(np.abs(audio))
    if peak > 0: audio = audio / peak
    if len(audio) >= N_SAMPLES:
        start = (len(audio) - N_SAMPLES) // 2
        audio = audio[start:start+N_SAMPLES]
    else:
        n_reps = (N_SAMPLES // len(audio)) + 2
        audio = np.tile(audio, n_reps)[:N_SAMPLES]
    return audio.astype(np.float32)

def extract_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP_LEN,
        n_mels=N_MEL, fmin=FMIN, fmax=FMAX)
    return librosa.power_to_db(mel + 1e-6, ref=np.max).astype(np.float32)

def normalize_mel(log_mel, norm_stats):
    mean = np.array(norm_stats['mel_mean'], dtype=np.float32)
    std  = np.array(norm_stats['mel_std'],  dtype=np.float32)
    std  = np.where(std < 1e-8, 1.0, std)
    log_mel = (log_mel - mean[:,None]) / std[:,None]
    if log_mel.shape[1] < TARGET_T:
        log_mel = np.pad(log_mel, ((0,0),(0,TARGET_T-log_mel.shape[1])), mode='edge')
    elif log_mel.shape[1] > TARGET_T:
        log_mel = log_mel[:,:TARGET_T]
    return log_mel.astype(np.float32)

def audio_to_tensor(path):
    audio    = preprocess_audio(path)
    log_mel  = extract_mel(audio)
    norm_mel = normalize_mel(log_mel, NORM_STATS)
    return torch.FloatTensor(norm_mel).unsqueeze(0).unsqueeze(0).to(DEVICE)

print("✅ Preprocessing functions defined")
print(f"  SR={SR} Hz · Duration={DURATION}s · N_mel={N_MEL} · T={TARGET_T} frames")
print(f"  Gate threshold  : {GATE_THRESHOLD}  (F1-optimised)")
print(f"  OOD conf  min   : {OOD_CONF_THR*100:.0f}%")
print(f"  OOD entropy max : {OOD_ENTROPY_THR}")


In [ ]:
# ── Cell 9: Production Inference Function ────────────────────────────────────
import torch, torch.nn.functional as F
import numpy as np

def run_cry_gate(audio_path):
    """Stage 1: Check if audio is a baby cry."""
    if _gate_model is None:
        return {'is_cry': None, 'score': 0.5, 'reason': 'gate not loaded'}
    try:
        x = audio_to_tensor(audio_path)
        with torch.no_grad():
            score = torch.sigmoid(_gate_model(x)).item()
        is_cry = score >= GATE_THRESHOLD
        return {'is_cry': is_cry, 'score': score,
                'reason': f"gate_score={score:.3f} {'≥' if is_cry else '<'} {GATE_THRESHOLD}"}
    except Exception as e:
        return {'is_cry': None, 'score': 0.5, 'reason': str(e)}

def predict(audio_path, verbose=False):
    """
    Run 3-stage production inference on a .wav / .mp3 file.
    Returns a result dict with prediction, confidence, probabilities, gate info.
    """
    print(f"\n{'='*55}")
    print(f"  🍼 Infant Cry Classifier v3 — 3-Stage Pipeline")
    print(f"{'='*55}")
    print(f"  Audio  : {audio_path}")
    print(f"  Device : {DEVICE}")
    print()

    if NORM_STATS is None:
        raise RuntimeError("norm_stats not loaded — run Cell 7 first")
    if not _loaded_models:
        raise RuntimeError("No models loaded — run Cell 6 first")

    # ── Stage 1: Cry Gate ────────────────────────────────────────────────────
    gate = run_cry_gate(audio_path)
    print(f"  Stage 1 — Gate  : score={gate['score']:.3f}  {gate['reason']}")
    if gate['is_cry'] is False:
        print(f"  🚫 NOT A CRY — rejected by cry gate")
        return {'is_cry': False, 'prediction': 'not_a_cry', 'confidence': 0.0,
                'gate_score': gate['score'], 'stage_blocked': 'cry_gate',
                'probabilities': {c: 0.0 for c in CLASSES}}

    # ── Preprocess ───────────────────────────────────────────────────────────
    x = audio_to_tensor(audio_path)

    # ── Stage 3 (run models first, OOD check after) ───────────────────────────
    all_probs = []
    with torch.no_grad():
        for model_obj, name in _loaded_models:
            raw_logits = model_obj(x).cpu()
            T = CALIB_TEMPERATURES.get(name, 1.0)
            probs = F.softmax(raw_logits / T, dim=1).numpy()[0]
            all_probs.append((probs, name))
            if verbose:
                top_cls = CLASSES[probs.argmax()]
                print(f"    {name:15s} → {top_cls:12s}  ({probs.max()*100:.1f}%)  T={T:.3f}")

    # Weighted ensemble with learned weights
    raw_w = np.array([CALIB_WEIGHTS.get(name, 0.25) for _, name in all_probs])
    w = raw_w / raw_w.sum()
    ensemble_probs = sum(wi * p for (p, _), wi in zip(all_probs, w))
    ensemble_probs = ensemble_probs / ensemble_probs.sum()

    max_conf = float(ensemble_probs.max())
    entropy  = float(-np.sum(ensemble_probs * np.log(ensemble_probs + 1e-9)))

    print(f"  Stage 2 — OOD   : conf={max_conf*100:.1f}%  entropy={entropy:.3f}")

    # ── Stage 2: OOD Check ───────────────────────────────────────────────────
    if max_conf < OOD_CONF_THR or entropy > OOD_ENTROPY_THR:
        print(f"  ⚠️  UNCERTAIN — OOD check failed")
        return {'is_cry': True, 'prediction': 'uncertain', 'confidence': max_conf*100,
                'entropy': entropy, 'gate_score': gate['score'],
                'stage_blocked': 'ood_check',
                'probabilities': {CLASSES[i]: round(float(ensemble_probs[i]*100),2) for i in range(5)}}

    pred_idx   = int(ensemble_probs.argmax())
    pred_cls   = CLASSES[pred_idx]
    confidence = float(ensemble_probs[pred_idx] * 100)
    reliability = 'HIGH' if confidence >= 80 else ('MEDIUM' if confidence >= 60 else 'LOW')

    print(f"  Stage 3 — Result: {pred_cls.upper()}")
    print()
    print(f"  {'='*51}")
    print(f"  ✅  PREDICTION  : {pred_cls.upper()}")
    print(f"     CONFIDENCE  : {confidence:.1f}%")
    print(f"     RELIABILITY : {reliability}")
    print(f"     ENTROPY     : {entropy:.4f}")
    print(f"     GATE SCORE  : {gate['score']:.3f}")
    print(f"     MODELS USED : {len(_loaded_models)}")
    print(f"  {'='*51}")
    print()
    print("  Per-class probabilities:")
    sorted_items = sorted(zip(CLASSES, ensemble_probs), key=lambda x: -x[1])
    for cls, p in sorted_items:
        bar = '█' * int(p * 100 // 5)
        print(f"    {cls:12s}: {p*100:5.1f}%  {bar}")

    return {
        'is_cry': True, 'prediction': pred_cls, 'confidence': round(confidence, 2),
        'reliability': reliability, 'entropy': round(entropy, 4),
        'gate_score': round(gate['score'], 4), 'stage_blocked': None,
        'n_models': len(_loaded_models),
        'probabilities': {CLASSES[i]: round(float(ensemble_probs[i]*100), 2) for i in range(5)},
    }

print("✅ predict() function ready")
print("   Usage: result = predict('path/to/cry.wav')")
print("   Usage: result = predict('path/to/cry.wav', verbose=True)")


In [ ]:
# ── Cell 10: Upload Audio and Classify ───────────────────────────────────────
from google.colab import files
import tempfile, os

print("Upload a .wav or .mp3 file:")
uploaded = files.upload()

for filename, data in uploaded.items():
    suffix = os.path.splitext(filename)[1] or '.wav'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
        tmp.write(data)
        tmp_path = tmp.name

    try:
        result = predict(tmp_path, verbose=True)
    finally:
        os.unlink(tmp_path)


In [ ]:
# ── Cell 11: Batch Inference on a Drive Folder ───────────────────────────────
import os
from pathlib import Path

# Set this to a Drive folder containing .wav files
AUDIO_FOLDER = "/content/drive/MyDrive/test_cries"

wav_files = sorted(Path(AUDIO_FOLDER).glob("**/*.wav")) if os.path.isdir(AUDIO_FOLDER) else []
print(f"Found {len(wav_files)} .wav files in {AUDIO_FOLDER}")
print()

results = []
for i, wav_path in enumerate(wav_files):
    print(f"[{i+1}/{len(wav_files)}] {wav_path.name}")
    try:
        res = predict(str(wav_path))
        results.append({'file': wav_path.name, **res})
    except Exception as e:
        print(f"  ❌ Error: {e}")
        results.append({'file': wav_path.name, 'error': str(e)})

# Summary table
print()
print("=" * 60)
print(f"  Batch Summary — {len(results)} files")
print("=" * 60)
for r in results:
    if 'error' in r:
        print(f"  ❌ {r['file']:30s}  ERROR: {r['error'][:30]}")
    elif not r.get('is_cry'):
        print(f"  🚫 {r['file']:30s}  not_a_cry (gate={r.get('gate_score',0):.2f})")
    else:
        print(f"  ✅ {r['file']:30s}  {r['prediction']:12s}  {r['confidence']:.1f}%  [{r['reliability']}]")


In [ ]:
# ── Cell 12: Production Model Metrics Summary ────────────────────────────────
print("="*62)
print("  🍼 NXM Infant Cry Classifier — Production v2 Metrics")
print("="*62)

metrics = {
    'BinaryCryCNN Gate' : {'acc': 98.56, 'f1': 0.9862, 'note': 'cry vs not-cry'},
    'BaselineCNN*'      : {'acc': 84.12, 'f1': 0.8157, 'note': 'T=0.873, w=13.7%'},
    'CNN+BiLSTM*'       : {'acc': 79.62, 'f1': 0.7915, 'note': 'T=0.951, w=16.2%'},
    'CryNet*'           : {'acc': 81.75, 'f1': 0.8164, 'note': 'T=1.002, w=34.1%'},
    'SE-ResNet*'        : {'acc': 85.38, 'f1': 0.8533, 'note': 'T=1.037, w=36.0% ⭐'},
}

print()
print(f"  {'Model':<22}  {'Val Acc':>8}  {'Macro F1':>9}  Note")
print("  " + "-"*60)
for name, m in metrics.items():
    print(f"  {name:<22}  {m['acc']:>7.2f}%  {m['f1']:>9.4f}  {m['note']}")

print("  " + "-"*60)
print(f"  {'Calibrated Ensemble':<22}  {'87.50':>7}%  {'0.8740':>9}  Phase 4 ✅")
print()
print("  Per-class F1 (production ensemble):")
per_class = {
    'belly_pain' : (0.8067, 'worst — confused with hungry (21 err)'),
    'burping'    : (0.8675, 'confused with tired (15 err)'),
    'discomfort' : (0.9565, 'best class ✅'),
    'hungry'     : (0.8537, 'high recall 89.4%'),
    'tired'      : (0.8856, 'high recall 94.4%'),
}
for cls, (f1, note) in per_class.items():
    bar = '█' * int(f1 * 20)
    print(f"  {cls:12s}: {f1:.4f}  {bar}  {note}")
print("="*62)
